# SAADE–AUA v9 | Незалежна перевірка AASIST / AASIST-L — FIXED v2

**Призначення:** перевірити відповідність виходів ONNX та еталонної PyTorch-моделі на однакових аудіосигналах. **Моделі не навчаються. Hidden test не читається. Основний експеримент v9 не змінюється.**

### Що виправлено після HTTP 404
У репозиторії Hugging Face `SpeechAntiSpoofingBenchmarks/AASIST` зараз немає `_net.py` та `aasist.py`, попри згадку в README. Тому ця версія бере **спільну архітектуру AASIST з оригінального `clovaai/aasist/models/AASIST.py`** (резерв — `_net.py` з AASIST-L), **параметри архітектури зі стандартних `AASIST.conf` та `AASIST-L.conf`**, а **ваги `.pth` з тих самих репозиторіїв HF**, що й файли ONNX у v9. Всі завантажені файли мають окремі SHA-256.

Офіційні джерела: [архітектура](https://github.com/clovaai/aasist/blob/main/models/AASIST.py), [конфігурація AASIST](https://github.com/clovaai/aasist/blob/main/config/AASIST.conf), [конфігурація AASIST-L](https://github.com/clovaai/aasist/blob/main/config/AASIST-L.conf); [ваги AASIST](https://huggingface.co/SpeechAntiSpoofingBenchmarks/AASIST/tree/main), [ваги AASIST-L](https://huggingface.co/SpeechAntiSpoofingBenchmarks/AASIST-L/tree/main).

**Правило прийняття:** строго завантажена PyTorch-архітектура ідентичної конфігурації, близькі обидва логіти, перевірений знак FAKE-score та звірка вже записаних прогнозів v9. Якщо хоч одна умова не підтверджена — статус `CHECK`; **жодного автоматичного переписування прогнозів або зміни порогів**.


## 1. Налаштування та залежності

In [ ]:
from pathlib import Path
import os, sys, json, math, hashlib, urllib.request, urllib.error, importlib.util, sqlite3, time
import numpy as np
import pandas as pd
import soundfile as sf
from scipy import signal
from fractions import Fraction
from sklearn.metrics import roc_auc_score
from IPython.display import display

PROJECT = Path('/home/jovyan/work/audio_real_fake')
BASE = PROJECT/'results'/'SAADE_AUA_2026_v9_COMPLETE_EVALUATION'
RUN = BASE/'FULL_EXPERIMENT'
CACHE = BASE/'pretrained_model_cache'
OUT = BASE/'AASIST_REFERENCE_PARITY_AUDIT'
OUT.mkdir(parents=True, exist_ok=True)
V6_DIR = next((p for p in [PROJECT/'results'/'SAADE_AUA_2026_v6_2_MATCHED_TTS',
                          PROJECT/'results'/'SAADE_AUA_2026_v6_MATCHED_TTS']
               if (p/'V6_LOCKED_MATCHED_SPLITS.csv').exists()), None)
assert V6_DIR is not None, 'Не знайдено matched REAL–TTS маніфест'
assert CACHE.is_dir() and (RUN/'03_MODEL_PROVENANCE.json').exists(), 'Потрібен завершений v9'
N_PAIRS = 12
MODELS = {
 'AASIST_ONNX': {'repo':'SpeechAntiSpoofingBenchmarks/AASIST',
                  'onnx':'aasist.onnx','weights':'AASIST.pth',
                  'model_config':{'architecture':'AASIST','nb_samp':64600,'first_conv':128,
                    'filts':[70,[1,32],[32,32],[32,64],[64,64]],
                    'gat_dims':[64,32], 'pool_ratios':[0.5,0.7,0.5,0.5],
                    'temperatures':[2.,2.,100.,100.]}},
 'AASIST_L_ONNX': {'repo':'SpeechAntiSpoofingBenchmarks/AASIST-L',
                  'onnx':'aasist-l.onnx','weights':'AASIST-L.pth',
                  'model_config':{'architecture':'AASIST','nb_samp':64600,'first_conv':128,
                    'filts':[70,[1,32],[32,32],[32,24],[24,24]],
                    'gat_dims':[24,32], 'pool_ratios':[0.4,0.5,0.7,0.5],
                    'temperatures':[2.,2.,100.,100.]}}
}
print('Перевірка тільки MATCHED_TRAIN. Жоден hidden-файл не читається.')


## 2. Безпечне відновлення реальних аудіопар з train

In [ ]:
manifest = pd.read_csv(V6_DIR/'V6_LOCKED_MATCHED_SPLITS.csv', low_memory=False)
train = manifest.loc[manifest['split'].astype(str).eq('train')].copy()
train['pair_id'] = train['pair_id'].astype(str)
train['label'] = train['label'].astype(int)
assert set(train['label']) == {0,1}
assert train.groupby('pair_id')['label'].agg(['sum','count']).eq([1,2]).all().all(), 'Неправильна структура пар'
pair_ids = np.random.default_rng(20261117).choice(
    np.asarray(sorted(train.pair_id.unique())), size=min(N_PAIRS,train.pair_id.nunique()), replace=False)
selected = train.loc[train.pair_id.isin(pair_ids)].copy()
assert len(selected) == 2*len(pair_ids)
assert not selected['split'].astype(str).eq('hidden_test').any()

def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(2*1024*1024), b''): h.update(b)
    return h.hexdigest()

def decode_exact_v9(path, start_sec, active_n):
    # Відтворюємо read_wave() із v9: завжди 4 с за 16 кГц, доповнення нулями до 64000.
    with sf.SoundFile(str(path)) as f:
        rate=f.samplerate
        f.seek(max(0,int(round(float(start_sec)*rate))))
        x=f.read(frames=int(math.ceil(4.0*rate)),dtype='float32',always_2d=True).mean(axis=1)
    if rate!=16000:
        ratio=Fraction(16000,rate)
        x=signal.resample_poly(x,ratio.numerator,ratio.denominator)
    x=np.nan_to_num(np.asarray(x,dtype=np.float32),nan=0,posinf=0,neginf=0)
    x=x[:int(active_n)]
    return np.pad(x,(0,max(0,64000-len(x))))[:64000].astype(np.float32)

samples=[]
for pid in pair_ids:
    duo=selected.loc[selected.pair_id.eq(pid)]
    starts=[]; available=[]
    for r in duo.itertuples(index=False):
        assert Path(r.path).is_file(),f'Аудіо відсутнє: {r.path}'
        with sf.SoundFile(str(r.path)) as f:
            available.append(float(len(f)/f.samplerate)-float(getattr(r,'segment_start_sec',0) or 0))
        starts.append(float(getattr(r,'segment_start_sec',0) or 0))
    active_n=min(64000,max(1,int(math.floor(min(available)*16000))))
    assert active_n>=8000,'Надто коротка пара для перевірки'
    for r in duo.itertuples(index=False):
        w=decode_exact_v9(r.path,float(getattr(r,'segment_start_sec',0) or 0),active_n)
        samples.append({'pair_id':pid,'label':int(r.label),
                        'sample_id':f'matched:{pid}:{int(r.label)}','wave':w,'active_n':active_n})
print('Діагностичних аудіозаписів:',len(samples), '| REAL:',sum(x['label']==0 for x in samples),
      '| FAKE:',sum(x['label']==1 for x in samples))


## 3. Офіційна PyTorch-архітектура та саме pretrained-вагове наповнення

Код архітектури беремо з офіційного GitHub (`models/AASIST.py`), а не з неіснуючого `AASIST/_net.py` у Hugging Face. Якщо GitHub недоступний, перевіряємо резервну копію того самого класу `Model` із репозиторію AASIST-L. **Завантаження ваг виконується окремо для кожної моделі.**


In [ ]:
import torch
import onnxruntime as ort

UPSTREAM_ARCHITECTURE_URLS = [
  'https://raw.githubusercontent.com/clovaai/aasist/main/models/AASIST.py',
  'https://huggingface.co/SpeechAntiSpoofingBenchmarks/AASIST-L/resolve/main/_net.py'
]

def download_url(url, destination, min_size=100):
    destination=Path(destination);destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.is_file() and destination.stat().st_size >= min_size:
        return destination
    req=urllib.request.Request(url,headers={
        'User-Agent':'SAADE-parity-audit/2.0', 'Accept-Encoding':'identity'})
    tmp=destination.with_name(destination.name+'.partial')
    try:
        with urllib.request.urlopen(req,timeout=180) as resp, open(tmp,'wb') as f:
            while True:
                block=resp.read(1024*1024)
                if not block:break
                f.write(block)
        if tmp.stat().st_size < min_size:
            raise RuntimeError(f'Завантажено неповний файл ({tmp.stat().st_size} байтів): {url}')
        if tmp.open('rb').read(32).lstrip().lower().startswith(b'<html'):
            raise RuntimeError('Замість коду/моделі прийшла HTML-сторінка: '+url)
        os.replace(tmp,destination)
    finally:
        tmp.unlink(missing_ok=True)
    return destination

def locate_official_architecture():
    root=OUT/'reference_sources'
    cached=root/'official_AASIST.py'
    if cached.exists() and cached.stat().st_size>=10000:
        code=cached.read_text(encoding='utf8')
        if 'class Model(' in code and 'class GraphAttentionLayer(' in code:
            source_note=root/'official_AASIST.source_url.txt'
            saved=source_note.read_text(encoding='utf8').strip() if source_note.exists() else 'cached_source_unknown_url'
            return cached,saved
    errors=[]
    for url in UPSTREAM_ARCHITECTURE_URLS:
        try:
            temporary=root/('official_AASIST.tmp.py')
            if temporary.exists():temporary.unlink()
            fetched=download_url(url,temporary,min_size=10000)
            code=fetched.read_text(encoding='utf8')
            if 'class Model(' not in code or 'class GraphAttentionLayer(' not in code:
                raise ValueError('Завантажений файл не містить очікуваних класів AASIST')
            compile(code,url,'exec')
            os.replace(fetched,cached)
            (root/'official_AASIST.source_url.txt').write_text(url,encoding='utf8')
            return cached,url
        except Exception as exc:
            errors.append(f'{url} -> {type(exc).__name__}: {exc}')
    raise RuntimeError('Не вдалося отримати еталонну архітектуру з ОБОХ адрес: \n'+'\n'.join(errors))

ARCH_FILE,ARCH_URL=locate_official_architecture()
module_name='_saade_official_upstream_aasist_architecture'
spec=importlib.util.spec_from_file_location(module_name, ARCH_FILE)
mod=importlib.util.module_from_spec(spec)
sys.modules[module_name]=mod
spec.loader.exec_module(mod)
assert hasattr(mod,'Model'), 'Немає класу Model у завантаженій архітектурі'
print('Архітектура PyTorch:',ARCH_URL)
print('SHA-256 архітектури:',sha256(ARCH_FILE))

def load_pytorch_reference(model_name,cfg):
    root=OUT/'reference_sources'/model_name
    weight_url=f'https://huggingface.co/{cfg["repo"]}/resolve/main/{cfg["weights"]}'
    weight=download_url(weight_url,root/cfg['weights'],min_size=100000)
    net=mod.Model(cfg['model_config'])
    state=torch.load(str(weight),map_location='cpu',weights_only=True)
    if isinstance(state,dict) and 'state_dict' in state:state=state['state_dict']
    if isinstance(state,dict) and 'model' in state and isinstance(state['model'],dict):state=state['model']
    if not isinstance(state,dict):raise RuntimeError('Checkpoint не містить state_dict: '+str(weight))
    if all(str(k).startswith('module.') for k in state):
        state={str(k)[len('module.'):]:v for k,v in state.items()}
    # Строге завантаження: невідповідність архітектури НЕ ігноруємо.
    net.load_state_dict(state,strict=True)
    net=net.eval().cpu().requires_grad_(False)
    return net, {'architecture_url':ARCH_URL,'architecture_sha256':sha256(ARCH_FILE),
                 'pytorch_weights_url':weight_url,'pth_sha256':sha256(weight),
                 'model_config':cfg['model_config']}

existing_provenance=json.loads((RUN/'03_MODEL_PROVENANCE.json').read_text(encoding='utf8'))
assert set(MODELS).issubset(existing_provenance), 'У v9 немає потрібних provenance даних AASIST'
print('Зафіксовані SHA-256 ONNX із v9:',{k:existing_provenance[k]['weights_sha256'] for k in MODELS})


## 4. Однаковий вхід → обидва виходи ONNX/PyTorch та ефект padding

Для кожного запису використовуються два 64 600-семплові сигнали: **нульове доповнення у v9** та **повторюване доповнення за оригінальним AASIST**. Обидві моделі отримують точно однаковий сигнал для кожного варіанта. Якщо parity не пройдено, файл зі статистикою все одно зберігається — без фальшивого `PASS`.


In [ ]:
def to_64600(w,active_n,method):
    if method=='v9_zeros':
        return np.pad(w[:64000],(0,600)).astype(np.float32)
    if method=='reference_repeat':
        x=w[:active_n]
        assert len(x)>0
        return np.tile(x,int(np.ceil(64600/len(x))))[:64600].astype(np.float32)
    raise ValueError(method)

def onnx_logits(session,x):
    meta=session.get_inputs()[0]
    if len(meta.shape)==2:inp=x[None,:]
    elif len(meta.shape)==3 and meta.shape[1] in (1,None,'1'):inp=x[None,None,:]
    else:raise RuntimeError('Непідтримуваний ONNX shape: '+str(meta.shape))
    outputs=session.run(None,{meta.name:inp})
    if len(outputs)==0:raise RuntimeError('ONNX не повернув виходів')
    logits=np.asarray(outputs[0],dtype=np.float64).reshape(-1)
    # Порівнюємо обидва класи; будь-який інший формат — CHECK, а не здогадка.
    if logits.size!=2:raise RuntimeError(f'Очікували 2 логіти AASIST, отримано {logits.size}')
    if not np.isfinite(logits).all():raise RuntimeError('ONNX повернув NaN/inf')
    return logits

rows=[]; summary=[]; ref_manifest={}
for name,cfg in MODELS.items():
    onnx_fp=CACHE/cfg['onnx']
    assert onnx_fp.is_file(),f'Не знайдено ONNX з v9: {onnx_fp}'
    actual_sha=sha256(onnx_fp)
    expected_sha=existing_provenance[name]['weights_sha256']
    assert actual_sha==expected_sha, f'ONNX модель змінилася з часу v9: {name}'
    net,meta=load_pytorch_reference(name,cfg)
    ref_manifest[name]={'onnx_sha256':actual_sha,**meta}
    session=ort.InferenceSession(str(onnx_fp),providers=['CPUExecutionProvider'])
    for s in samples:
        for padding in ('v9_zeros','reference_repeat'):
            x=to_64600(s['wave'],s['active_n'],padding)
            y_onnx=onnx_logits(session,x)
            with torch.inference_mode():
                raw=net(torch.from_numpy(x)[None,:])
                y_pt=(raw[-1] if isinstance(raw,(list,tuple)) else raw)
            y_pt=np.asarray(y_pt.detach().cpu().numpy(),dtype=np.float64).reshape(-1)
            assert y_pt.size==2 and np.isfinite(y_pt).all(), 'PyTorch вихід не має 2 логітів'
            rows.append(dict(model=name,pair_id=s['pair_id'],label=s['label'],sample_id=s['sample_id'],
                padding=padding,onnx_bonafide=float(y_onnx[1]),pytorch_bonafide=float(y_pt[1]),
                onnx_fake_score=float(-y_onnx[1]),pytorch_fake_score=float(-y_pt[1]),
                abs_diff=float(abs(y_onnx[1]-y_pt[1])),
                class0_abs_diff=float(abs(y_onnx[0]-y_pt[0])),
                crossed_pairing_error=float(abs(y_onnx[0]-y_pt[1])+abs(y_onnx[1]-y_pt[0])),
                direct_pairing_error=float(abs(y_onnx[0]-y_pt[0])+abs(y_onnx[1]-y_pt[1])),
                onnx_class0=float(y_onnx[0]),torch_class0=float(y_pt[0]),active_n=s['active_n']))
    del net,session
    print(name,'перевірено. SHA-256 PyTorch checkpoint:',meta['pth_sha256'])

(OUT/'00_REFERENCE_PROVENANCE.json').write_text(json.dumps(ref_manifest,indent=2,ensure_ascii=False),encoding='utf8')
result=pd.DataFrame(rows)
result.to_csv(OUT/'01_ONNX_PYTORCH_PARITY_PER_RECORD.csv',index=False)
from scipy.stats import spearmanr
for (model,padding),d in result.groupby(['model','padding']):
    a=d['onnx_bonafide'].to_numpy(); b=d['pytorch_bonafide'].to_numpy()
    rho=float(spearmanr(a,b).statistic) if len(np.unique(a))>1 and len(np.unique(b))>1 else float('nan')
    direct=float(d['direct_pairing_error'].mean())
    crossed=float(d['crossed_pairing_error'].mean())
    close1=bool(np.allclose(a,b,rtol=1e-2,atol=2e-2))
    close0=bool(np.allclose(d['onnx_class0'],d['torch_class0'],rtol=1e-2,atol=2e-2))
    correct_order=bool(direct<crossed)
    pass_status=bool(close1 and close0 and correct_order and np.isfinite(rho) and rho>0.999)
    summary.append(dict(model=model,padding=padding,N=len(d),
                        max_abs_diff_class1=float(d.abs_diff.max()),
                        max_abs_diff_class0=float(d.class0_abs_diff.max()),
                        spearman=rho,outputs_numerically_close=bool(close1 and close0),
                        class_order_preserved=correct_order,status='PASS' if pass_status else 'CHECK'))
report=pd.DataFrame(summary)
report.to_csv(OUT/'02_PARITY_SUMMARY.csv',index=False)
display(report)
if (report.status=='PASS').all():
    print('PASS: виходи ONNX та PyTorch збігаються для однакових сигналів.')
else:
    print('CHECK: виходи не збігаються. Не відкривайте hidden test, доки причину не встановлено.')


## 5. Вплив типу padding і наявність старих прогнозів у v9

In [ ]:
pad=result.pivot_table(index=['model','sample_id','label'],columns='padding',values='onnx_fake_score').reset_index()
pad['abs_padding_effect']=(pad['v9_zeros']-pad['reference_repeat']).abs()
impact=pad.groupby('model')['abs_padding_effect'].agg(['count','median','max']).reset_index()
impact.columns=['model','n','median_abs_change','max_abs_change']
cal_path=RUN/'04_CALIBRATION.json'
if cal_path.exists():
    calibration=json.loads(cal_path.read_text(encoding='utf8'))
    for model,cal in calibration.items():
        if model in MODELS and 'threshold' in cal:
            mask=pad.model.eq(model); threshold=float(cal['threshold'])
            pad.loc[mask,'prediction_changed']=(pad.loc[mask,'v9_zeros']>=threshold).astype(int).values != (pad.loc[mask,'reference_repeat']>=threshold).astype(int).values
            impact.loc[impact.model.eq(model),'changed_decisions']=int(pad.loc[mask,'prediction_changed'].sum())
impact.to_csv(OUT/'03_PADDING_EFFECT.csv',index=False)
pad.to_csv(OUT/'03_PADDING_EFFECT_PER_RECORD.csv',index=False)
display(impact)

db_path=RUN/'05_FULL_PREDICTIONS.sqlite3'
oldcheck=pd.DataFrame(columns=['model','sample_id','existing_score_fake','recomputed_score_fake','abs_diff'])
if db_path.exists():
    con=sqlite3.connect(f'file:{db_path}?mode=ro',uri=True)
    checks=[]
    for s in samples:
        for name in MODELS:
            old=con.execute('SELECT score_fake FROM predictions WHERE model=? AND cohort=? AND sample_id=? AND scenario=?',
                (name,'MATCHED_TRAIN',s['sample_id'],'clean')).fetchone()
            if old is not None:
                new=pad.loc[(pad.model==name)&(pad.sample_id==s['sample_id']),'v9_zeros'].iloc[0]
                checks.append({'model':name,'sample_id':s['sample_id'],'existing_score_fake':old[0],
                               'recomputed_score_fake':float(new),'abs_diff':abs(old[0]-new)})
    con.close()
    oldcheck=pd.DataFrame(checks) if checks else oldcheck
else:
    print('SQLite недоступний: наявні v9-прогнози додатково не звірені.')
oldcheck.to_csv(OUT/'04_EXISTING_V9_SCORE_CHECK.csv',index=False)
if len(oldcheck):
    display(oldcheck.groupby('model').abs_diff.agg(['count','max']))
    consistent_v9=bool(oldcheck.abs_diff.max()<0.02)
else:
    consistent_v9=False
    print('У SQLite не знайдено вихідних оцінок для обраних 12 пар; це не підтвердження відсутності помилок.')

pass_all=bool((report.status=='PASS').all() and consistent_v9)
summary={'overall_status':'PASS' if pass_all else 'CHECK',
         'pytorch_onnx_parity_pass':bool((report.status=='PASS').all()),
         'existing_v9_scores_match':consistent_v9,
         'padding_changed_decisions':{str(x.model):int(x.changed_decisions) if pd.notna(x.changed_decisions) else None
                                      for x in impact.itertuples(index=False)} if 'changed_decisions' in impact else {},
         'note':'Порівняння типів padding не є автоматичною підставою змінювати v9. Відхилення потребують аналізу.'}
(OUT/'05_AUDIT_DECISION.json').write_text(json.dumps(summary,indent=2,ensure_ascii=False),encoding='utf8')
print('РЕЗУЛЬТАТ:',summary['overall_status'])
print('Файли перевірки:',OUT/'02_PARITY_SUMMARY.csv',OUT/'03_PADDING_EFFECT.csv')
print('Основний v9 не змінено. Hidden не відкрито.')
